# XMCA-Net-Lite: 01 - Evidence Corpus & FAISS Index
### Purpose
Construct the unified knowledge evidence corpus from dataset ground truth rationales and domain knowledge. Build a dense FAISS index using BGE-small / E5-base embeddings.


In [ ]:
import sys, json
from pathlib import Path
PROJECT_ROOT = Path("/kaggle/working/xmca") if Path("/kaggle/working").exists() else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed
from src.retrieval.corpus import build_evidence_corpus
from src.retrieval.indexer import DenseRetriever
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seed = config["project"]["default_seed"]
set_seed(seed)


In [ ]:
splits_dir = Path(config["paths"]["splits_dir"])
corpora_dir = Path(config["paths"]["corpora_dir"])

# Ingest all training split records to extract knowledge evidence
training_records = []
for p in splits_dir.glob("*/*train.json"):
    with open(p, "r", encoding="utf-8") as f:
        training_records.extend(json.load(f))

print(f"Ingesting evidence from {len(training_records)} training records...")
corpus_file = corpora_dir / "evidence_corpus.jsonl"
corpus_entries = build_evidence_corpus(training_records, corpus_file)
print(f"Constructed evidence corpus with {len(corpus_entries)} passages.")


In [ ]:
# Build and persist FAISS index
retriever = DenseRetriever(model_id=config["models"]["retriever"]["model_id"])
retriever.build_index(corpus_entries)

index_file = corpora_dir / "faiss_index.bin"
meta_file = corpora_dir / "corpus_metadata.json"
retriever.save(index_file, meta_file)

# Test sample retrieval
query = "What furniture or objects are present in the room?"
results = retriever.search(query, top_k=3)
print(f"\nSample query: '{query}'")
for doc, score in results:
    print(f"  [Score: {score:.4f}] {doc['text']}")

save_run_metadata(corpora_dir, config, seed, "01_prepare_corpora", {"corpus_size": len(corpus_entries)})
print("\nStep 01 completed successfully. FAISS index saved.")
